In [ ]:
import sys
import os

# 1. Determinar la ruta absoluta de la carpeta raíz (un nivel por encima de 'notebooks')
ruta_raiz = os.path.abspath(os.path.join('..'))

# 2. Añadir la ruta raíz a las variables de entorno para importar módulos 'src'
if ruta_raiz not in sys.path:
    sys.path.append(ruta_raiz)

# 3. Cambiar el directorio de trabajo del cuaderno a la raíz del proyecto para leer 'data/'
os.chdir(ruta_raiz)

print(f"Directorio de trabajo actual: {os.getcwd()}")

### Función para cálculo de embeddings por cada párrafo de un texto

In [ ]:
import os
from getpass import getpass
from pathlib import Path
# Importaremos tu función (que contendrá la lógica de lotes de 100) desde el módulo externo
from src.embeddings import procesar_markdown_a_embeddings

# 1. Autenticación segura de la API
print("--- AUTENTICACIÓN OPENAI ---")
if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("Introduce tu OpenAI API Key (quedará oculta): ")

# 2. Configuración de directorios
directorio_leyes_md = Path("data/02_interim")
directorio_salida = Path("data/04_embeddings")
directorio_salida.mkdir(parents=True, exist_ok=True)

print("\n--- CALCULANDO EMBEDDINGS DE LAS LEYES ---")

# 3. Procesamiento iterativo de todas las leyes disponibles
archivos_md = list(directorio_leyes_md.glob("*.md"))

if not archivos_md:
    print(f"✗ Error: No se encontraron archivos .md en {directorio_leyes_md}.")
    print("Asegúrate de haber ejecutado el Paso 1 del Cuaderno 01 para extraer los PDFs.")
else:
    for ruta_md in archivos_md:
        # Generamos dinámicamente la ruta de guardado basada en el nombre del documento
        nombre_archivo_pkl = f"embeddings_ley_{ruta_md.stem}.pkl"
        ruta_pkl = directorio_salida / nombre_archivo_pkl
        
        # Validación de preexistencia para ahorrar costes de API
        if ruta_pkl.exists():
            print(f"⏭ Saltando '{ruta_md.name}': El archivo de embeddings ya existe.")
            continue

        # Llamada a la función externa que ejecutará tu lógica de procesamiento en lotes
        procesar_markdown_a_embeddings(
            ruta_md=ruta_md,
            ruta_pkl=ruta_pkl,
            modelo="text-embedding-3-small"
        )

### Función para calcular los cosenos de similitud y guardarlos en un archivo .csv

In [ ]:
from pathlib import Path
# Importaremos la lógica matemática desde el módulo externo
from src.similarity import procesar_similitudes_ley

# 4. Configuración de umbrales
# >>> ACCIÓN REQUERIDA: Introduce aquí los umbrales óptimos (F1 o F2) que decidiste en el Cuaderno 02
umbrales_aplicar = {
    "dimensión1": 0.4452,     # Sustituye este valor por el tuyo
    "dimensión2": 0.4790   # Sustituye este valor por el tuyo
}

# 5. Configuración de directorios
directorio_embeddings = Path("data/04_embeddings")
directorio_salida = Path("data/05_processed")
directorio_salida.mkdir(parents=True, exist_ok=True)

print("--- CALCULANDO SIMILITUDES Y EXPORTANDO RESULTADOS ---")

# 6. Identificar las leyes procesadas
archivos_leyes = list(directorio_embeddings.glob("embeddings_ley_*.pkl"))

if not archivos_leyes:
    print(f"✗ Error: No se encontraron embeddings de leyes en {directorio_embeddings}.")
else:
    for ruta_ley in archivos_leyes:
        print(f"\nCruzando datos para la ley: {ruta_ley.name}")
        
        # 7. Iterar sobre las dimensiones definidas en los umbrales
        for dimension, umbral in umbrales_aplicar.items():
            ruta_actividades = directorio_embeddings / f"embeddings_actividades_{dimension}.pkl"
            
            if not ruta_actividades.exists():
                print(f"  ⚠ Advertencia: No se encontró {ruta_actividades.name}. Saltando dimensión '{dimension}'.")
                continue
            
            # Generación de rutas dinámicas seguras
            # Extraemos el nombre base de la ley (ej: embeddings_ley_ley_aguas.pkl -> ley_aguas)
            nombre_ley = ruta_ley.stem.replace("embeddings_ley_", "")
            archivo_salida = directorio_salida / f"{nombre_ley}_matches_original_{dimension}.csv"
            
            # 8. Llamada a la función externa para procesar la matriz y generar el CSV
            procesar_similitudes_ley(
                ruta_ley_pkl=ruta_ley,
                ruta_act_pkl=ruta_actividades,
                umbral=umbral,
                ruta_salida=archivo_salida
            )

### Función para eliminación de emparejamientos con párrafos de menos de 10 palabras

In [ ]:
import pandas as pd
from pathlib import Path

print("--- FILTRANDO PÁRRAFOS CORTOS (< 10 PALABRAS) ---")
directorio_procesado = Path("data/05_processed")

# Buscar los archivos originales recién creados
archivos_originales = list(directorio_procesado.glob("*_matches_original_*.csv"))

if not archivos_originales:
    print("✗ Error: No se encontraron archivos originales para limpiar.")
else:
    for ruta_csv in archivos_originales:
        archivo_salida = Path(str(ruta_csv).replace("_original_", "_limpio_"))
        
        if archivo_salida.exists():
            print(f"⏭ Saltando: '{archivo_salida.name}' ya existe.")
            continue
            
        # Cargar el CSV original
        df = pd.read_csv(ruta_csv, sep=';', encoding='utf-8-sig')
        filas_antes = len(df)
        
        # Filtrar filas donde la longitud de la columna de texto (separada por espacios) sea >= 10
        # Asumimos que la columna se llama "Párrafo de la Ley" (como definiste en tu código)
        df_limpio = df[df["Párrafo de la Ley"].apply(lambda x: len(str(x).split()) >= 10)]
        filas_despues = len(df_limpio)
        
        # Guardar el nuevo CSV
        df_limpio.to_csv(archivo_salida, index=False, sep=';', encoding='utf-8-sig')
        
        print(f"✓ Limpiado: {ruta_csv.name}")
        print(f"  > Filas originales: {filas_antes} | Filas conservadas: {filas_despues} | Descartadas: {filas_antes - filas_despues}")

### Función para cálculo de dirección de emparejamientos

In [ ]:
from pathlib import Path
import os
# Importaremos la lógica de conexión a la API y clasificación desde el módulo externo
from src.classification import procesar_clasificacion_direccional

# 9. Configuración de Prompts por Dimensión
# >>> ACCIÓN REQUERIDA: Crea un archivo .txt por cada dimensión en la carpeta data/01_raw/.
# REGLA ESTRICTA PARA EL USUARIO: El texto de esos archivos debe terminar obligatoriamente con:
# "Devuelve única y exclusivamente un objeto JSON con dos claves. 
# 'razonamiento': Breve justificación que conecte la directriz con el impacto (máx 25 palabras).
# 'resultado': El número entero 1, 0, o -1."
archivos_prompts = {
    "dimensión1": Path("data/01_raw/prompt_dimensión1.txt"),  # Sustituye prompt_dimensión1 por el nombre de tu archivo de prompt
    "dimensión2": Path("data/01_raw/prompt_dimensión2.txt")   # Sustituye prompt_dimensión2 por el nombre de tu archivo de prompt
}

# 10. Procesamiento de la Dirección (Clasificación con LLM)
print("--- INICIANDO CLASIFICACIÓN DIRECCIONAL CON LLM ---")
directorio_procesado = Path("data/05_processed")

# Buscamos exclusivamente los CSV que ya pasaron por el filtro de 10 palabras (limpios)
archivos_matches = list(directorio_procesado.glob("*_matches_limpio_*.csv"))

if not archivos_matches:
    print(f"✗ Error: No se encontraron archivos limpios de similitud en {directorio_procesado}.")
else:
    for ruta_csv in archivos_matches:
        # Extraer dinámicamente la dimensión (ej. ley_aguas_matches_limpio_economia.csv -> economia)
        dimension = ruta_csv.stem.split("_matches_limpio_")[-1]
        
        if dimension not in archivos_prompts:
            print(f"  ⚠ Advertencia: No hay prompt configurado para '{dimension}'. Saltando...")
            continue
            
        ruta_prompt = archivos_prompts[dimension]
        if not ruta_prompt.exists():
            print(f"  ✗ Error: El archivo de prompt {ruta_prompt} no existe.")
            continue

        # Definir archivo de salida final cambiando 'limpio' por 'direccion'
        archivo_salida = Path(str(ruta_csv).replace("_limpio_", "_direccion_"))
        
        # Sistema de reanudación: si la API se cortó o ya se hizo, no repetimos el gasto
        if archivo_salida.exists():
            print(f"⏭ Saltando '{ruta_csv.name}': La clasificación ya fue realizada.")
            continue
        
        print(f"\nClasificando direcciones para: {ruta_csv.name}")
        
        # Llamada a la función externa que encapsulará la iteración del DataFrame y la API
        procesar_clasificacion_direccional(
            ruta_csv_entrada=ruta_csv,
            ruta_prompt=ruta_prompt,
            ruta_csv_salida=archivo_salida,
            api_key=os.environ.get("OPENAI_API_KEY"),
            modelo="gpt-4o"
        )

### Función para realizar operaciones y cálculo de índice

In [ ]:
from pathlib import Path
# Importaremos la lógica matemática desde src/metrics.py (cuando la construyamos)
from src.metrics import calcular_indices_finales

# 11. Configuración de parámetros para los índices
# >>> ACCIÓN REQUERIDA: Introduce aquí los valores exactos para cada dimensión
config_indices = {
    "edimensión1": {
        "valor_actividades": 51,  # Sustituye por el número total de actividades de esta dimensión
        "valor_umbral": 0.1982    # Sustituye por el valor real del umbral al cuadrado
    },
    "dimensión2": {
        "valor_actividades": 161,  # Sustituye por el número total de actividades de esta dimensión
        "valor_umbral": 0.2294    # Sustituye por el valor real del umbral al cuadrado
    }
}

print("--- CALCULANDO ÍNDICES Y MÉTRICAS FINALES ---")
directorio_procesado = Path("data/05_processed")

# Buscamos exclusivamente los archivos que ya pasaron por el LLM (dirección)
archivos_direccion = list(directorio_procesado.glob("*_matches_direccion_*.csv"))

if not archivos_direccion:
    print("✗ Error: No se encontraron archivos de dirección para calcular los índices.")
else:
    for ruta_csv in archivos_direccion:
        # Extraer la dimensión dinámicamente del nombre del archivo
        # (ej. ley_aguas_matches_direccion_economia.csv -> economia)
        dimension = ruta_csv.stem.split("_matches_direccion_")[-1]
        
        if dimension not in config_indices:
            print(f"  ⚠ Advertencia: No hay configuración de índices para '{dimension}'. Saltando...")
            continue
            
        archivo_salida = Path(str(ruta_csv).replace("_direccion_", "_indice_"))
        
        # Punto de control para evitar reprocesamiento
        if archivo_salida.exists():
            print(f"⏭ Saltando '{ruta_csv.name}': El cálculo de índices ya existe.")
            continue
            
        print(f"\nProcesando índices para: {ruta_csv.name}")
        
        # Extraer los parámetros específicos de la dimensión actual
        parametros = config_indices[dimension]
        
        # Llamada a la función externa que encapsulará el mapeo numérico y las operaciones
        calcular_indices_finales(
            ruta_csv_entrada=ruta_csv,
            ruta_csv_salida=archivo_salida,
            valor_actividades=parametros["valor_actividades"],
            valor_umbral=parametros["valor_umbral"]
        )